#### Average Close and Volume for Each Stock: Calculate the average closing price and volume for each stock. Provide insights into the overall performance and trading activity of each stock. (Trade Data) 
###

In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.window import Window
from pyspark.sql.functions import *


df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

window_spec = Window.partitionBy(col('Name'))
df1 = df.withColumn('avg_close' , round(avg(col('Volume')).over(window_spec),2) )
df2 = df1.withColumn('avg_volume' , round(avg(col('close')).over(window_spec),2) )
df2.show()

### Highest Closing Price of Each Quarter for Each Calendar Year: Identify and extract the highest closing price for each quarter of each calendar year. This will help investors understand the quarterly performance trends. (Trade Data) ###


In [0]:
# df.printSchema()
# print(df.dtypes)
# df['date'].dataType
# df.withColumn('date' , to_date(col('date')) )

from pyspark.sql.functions import *
from pyspark.sql.window import Window

df1 = df.withColumn(
    "Quarter",
     when(month(col("Date")).isin(1,2,3), 1)
    .when(month(col("Date")).isin(4,5,6), 2)
    .when(month(col("Date")).isin(7,8,9), 3)
    .when(month(col("Date")).isin(10,11,12), 4)
)
df1 = df1.withColumn('Year' , year(col('date')))

df2  = df1.groupBy(col('Year') , col('Quarter')).agg(max(col('close')).alias('Highest Closing Price'))

display(df2)

###Bar Graph for Daily Price Gap Analysis: Create a bar graph to visually analyze the daily price gap, which is the difference between the previous day's close and the current day's open. This will provide insights into potential market gaps and their impact on stock prices. For AAL Stock. (Trade Data) ###

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

In [0]:
window_spec = Window.partitionBy().orderBy(col('date'))
df = df.withColumn( 'prev_close' , lag(col('close')).over(window_spec) )
df = df.withColumn('difference', (col('close') - col('prev_close')) )
display(df)

###Identify in which year maximum number of customer become the member and among those which customer did most transaction. (Starbucks Data) 
###

In [0]:
df = spark.read.format('csv').option('inferschema', True).option('header',True).load('/Volumes/workspace/default/assignment_7/profile.csv')

In [0]:
df = df.withColumn('date' , to_date(col('became_member_on') , 'yyyyMMdd')) 
df = df.groupBy(year(col('date'))).agg(count(col('id')).alias('Total Counts'))
display(df)

###Calculate average Price-to-Earnings (P/E) Ratio for each Stock: Create a new column named pe_ratio representing the Price-to-Earnings ratio for each stock on a given day. The P/E ratio is calculated as the closing price divided by the earnings per share (EPS). Assume a constant EPS for simplicity. Use EPS = 5.0. (Trade Data) 
###

In [0]:
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

In [0]:
df = df.withColumn('earning', round((col('high') - col('low') ),3).alias('Earnings') )
df = df.withColumn('P/E Ratio' , round(( (col('close'))/col('earning') ), 3).alias('EPS') )

display(df)

###Moving Average Trend Analysis: Implement a transformation to calculate the 20-day moving average (ma_20) for the closing prices of each stock. This moving average will be used to identify potential trends and smooth out short-term fluctuations. (Trade Data) 


In [0]:
from pyspark.sql.windows import Window

df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/assignment_7/all_stocks_5yr.csv')

window_spec = Window.partitionBy(col('Name')).orderBy(col('close'))

df = df.withColumn('moving average' , avg(col('close')) over(col('')) )

df.show()

###Moving Average Trend Analysis: Implement a transformation to calculate the 20-day moving average (ma_20) for the closing prices of each stock. This moving average will be used to identify potential trends and smooth out short-term fluctuations. (Trade Data) 

###

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

windowSpec = Window.partitionBy("name").orderBy("date").rowsBetween(-19,0)

ma_df = df.withColumn(
    "ma_20",
    avg("close").over(windowSpec)
)

display(ma_df)

##### Date with Highest Intraday Profit Each Month: Determine, for each month, the date where a person would have earned the highest intraday profit. Intraday profit is calculated as the difference between the opening and closing prices. (Trade Data) 

In [0]:
df = df.withColumn("Intraday_Profit", abs(col("close") - col("open")))
window_spec = Window.partitionBy(month(col("date"))).orderBy(col("Intraday_Profit").desc())
df_highest_intraday = df.withColumn("rank", rank().over(window_spec)).filter(col("rank") == 1).select("date", "Intraday_Profit")
display(df_highest_intraday)

##### Delete the unwanted column from all three tables then explode the value column from transcribe table (Note: the value column is having offer id as offer_id and offer id handle both the cases also cache the dataframe returned after adding these three new columns) and find all the male customer who completed bogo offer and earned a reward of 5 or more. Also From which customer starbucks earned maximum profit. (Starbucks Data) 

In [0]:


profile_df = profile_df.drop("unwanted_column")
transcript_df = transcript_df.drop("unwanted_column")
offer_df = offer_df.drop("unwanted_column")
transcript_df = transcript_df.withColumn("offer_id", explode(col("value.offer_id")))
transcript_df.cache()
male_bogo = transcript_df.join(profile_df, "customer_id").join(offer_df, "offer_id") \
    .filter((col("gender") == "M") & (col("offer_type") == "bogo") & (col("reward") >= 5) & (col("event") == "completed"))
display(male_bogo)
profit_df = transcript_df.groupBy("customer_id").agg(sum("reward").alias("total_profit"))
max_profit_customer = profit_df.orderBy(col("total_profit").desc()).limit(1)
display(max_profit_customer)

##### Find all the customers who did not received any offer they just performed some transactions. (Starbucks Data) 


In [0]:
no_offer_customers = transcript_df.filter(col("event") == "transaction").join(offer_df, transcript_df.offer_id == offer_df.offer_id, "left_anti")
display(no_offer_customers.select("customer_id").distinct())

##### Find the number of completed offers for each channel and their respective total rewards. (Starbucks Data) 


In [0]:
completed_offers = transcript_df.filter(col("event") == "completed").join(offer_df, "offer_id")
channel_stats = completed_offers.groupBy("channel").agg(count("*").alias("completed_offers"), sum("reward").alias("total_rewards"))
display(channel_stats)

#### Total Earnings Calculation: Calculate the total earnings of a person assuming they buy shares on the first working day of each month and sell on the last working day of the month at the lowest and highest prices, respectively. Summarize the total earnings for analysis and Find which Share has given the highest Profit. (Trade Data)



In [0]:
from pyspark.sql.functions import *

window_spec = Window.partitionBy(year(col("date")), month(col("date")), col("Name")).orderBy(col("date"))

monthly_df = df.withColumn("first_day", first(col("date")).over(window_spec)) \
    .withColumn("last_day", last(col("date")).over(window_spec))

earnings_df = monthly_df.groupBy("Name", year(col("date")).alias("Year"), month(col("date")).alias("Month")) \
    .agg(min("low").alias("Buy_Price"), max("high").alias("Sell_Price")) \
    .withColumn("Earnings", col("Sell_Price") - col("Buy_Price"))

total_earnings = earnings_df.groupBy("Name").agg(sum("Earnings").alias("Total_Earnings"))

display(total_earnings.orderBy(col("Total_Earnings").desc()))

#### Identify the most popular offer among female customers who have an income of more than 50,000. (Perform broadcast join) (Starbucks Data) ###

In [0]:
from pyspark.sql.functions import broadcast
female_high_income = profile_df.filter((col("gender") == "F") & (col("income") > 50000))
popular_offer = transcript_df.join(broadcast(female_high_income), "customer_id") \
    .filter(col("event") == "completed") \
    .groupBy("offer_id").agg(count("*").alias("completed_count")) \
    .orderBy(col("completed_count").desc()).limit(1)
display(popular_offer)